In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Iterable

from PIL import Image, ImageDraw, ImageFont


OUTPUT_DIR = Path("images")

BACKGROUND_COLOR = "#2563EB"
TEXT_COLOR = "#FFFFFF"

# A square master image gives the best results when resizing.
MASTER_SIZE = 1024


def find_font() -> str | None:
    """
    Find a bold sans-serif font on common operating systems.

    Add another font path here if you want to use a specific font.
    """
    possible_fonts = [
        # Linux
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation2/LiberationSans-Bold.ttf",

        # macOS
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf",
        "/System/Library/Fonts/Supplemental/Helvetica.ttc",

        # Windows
        r"C:\Windows\Fonts\arialbd.ttf",
        r"C:\Windows\Fonts\segoeuib.ttf",
    ]

    for font_path in possible_fonts:
        if Path(font_path).exists():
            return font_path

    return None


def load_font(size: int) -> ImageFont.FreeTypeFont | ImageFont.ImageFont:
    font_path = find_font()

    if font_path:
        return ImageFont.truetype(font_path, size=size)

    print("Warning: bold system font not found. Using Pillow's default font.")
    return ImageFont.load_default()


def create_master_icon() -> Image.Image:
    """
    Create a high-resolution circular SD favicon.
    """
    image = Image.new(
        mode="RGBA",
        size=(MASTER_SIZE, MASTER_SIZE),
        color=(0, 0, 0, 0),
    )

    draw = ImageDraw.Draw(image)

    margin = 32
    draw.ellipse(
        (
            margin,
            margin,
            MASTER_SIZE - margin,
            MASTER_SIZE - margin,
        ),
        fill=BACKGROUND_COLOR,
    )

    text = "SD"
    font = load_font(size=430)

    # Measure the rendered text.
    bbox = draw.textbbox((0, 0), text, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]

    # Center text visually inside the circle.
    x = (MASTER_SIZE - text_width) / 2 - bbox[0]
    y = (MASTER_SIZE - text_height) / 2 - bbox[1] - 18

    draw.text(
        (x, y),
        text,
        font=font,
        fill=TEXT_COLOR,
    )

    return image


def save_png_sizes(
    master: Image.Image,
    sizes: Iterable[int],
) -> None:
    for size in sizes:
        resized = master.resize(
            (size, size),
            Image.Resampling.LANCZOS,
        )

        output_path = OUTPUT_DIR / f"favicon-{size}x{size}.png"
        resized.save(output_path, format="PNG", optimize=True)
        print(f"Created {output_path}")


def save_apple_touch_icon(master: Image.Image) -> None:
    size = 180

    icon = master.resize(
        (size, size),
        Image.Resampling.LANCZOS,
    )

    output_path = OUTPUT_DIR / "apple-touch-icon-180x180.png"
    icon.save(output_path, format="PNG", optimize=True)
    print(f"Created {output_path}")


def save_ico(master: Image.Image) -> None:
    """
    Save several resolutions inside a single ICO file.
    """
    ico_sizes = [(16, 16), (32, 32), (48, 48), (64, 64)]

    output_path = OUTPUT_DIR / "favicon.ico"

    master.save(
        output_path,
        format="ICO",
        sizes=ico_sizes,
    )

    print(f"Created {output_path}")


def save_svg() -> None:
    """
    Create a scalable SVG version of the favicon.
    """
    svg_content = f"""<svg
    xmlns="http://www.w3.org/2000/svg"
    viewBox="0 0 512 512"
    role="img"
    aria-labelledby="title"
>
  <title id="title">Shayan Dodge SD favicon</title>

  <circle
      cx="256"
      cy="256"
      r="240"
      fill="{BACKGROUND_COLOR}"
  />

  <text
      x="256"
      y="277"
      text-anchor="middle"
      dominant-baseline="middle"
      font-family="Arial, Helvetica, sans-serif"
      font-size="220"
      font-weight="700"
      letter-spacing="-18"
      fill="{TEXT_COLOR}"
  >
    SD
  </text>
</svg>
"""

    output_path = OUTPUT_DIR / "favicon.svg"
    output_path.write_text(svg_content, encoding="utf-8")
    print(f"Created {output_path}")


def save_manifest() -> None:
    manifest = {
        "name": "Shayan Dodge",
        "short_name": "SD",
        "icons": [
            {
                "src": "/images/favicon-192x192.png",
                "sizes": "192x192",
                "type": "image/png",
            },
            {
                "src": "/images/favicon-512x512.png",
                "sizes": "512x512",
                "type": "image/png",
            },
        ],
        "theme_color": BACKGROUND_COLOR,
        "background_color": "#FFFFFF",
        "display": "standalone",
        "start_url": "/",
    }

    output_path = OUTPUT_DIR / "manifest.json"

    output_path.write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )

    print(f"Created {output_path}")


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    master = create_master_icon()

    save_png_sizes(
        master,
        sizes=[16, 32, 192, 512],
    )

    save_apple_touch_icon(master)
    save_ico(master)
    save_svg()
    save_manifest()

    print("\nAll favicon files were generated successfully.")
    print(f"Output folder: {OUTPUT_DIR.resolve()}")


if __name__ == "__main__":
    main()